# Fraud Mitigation Agent · 05 Behavior Analysis

Comparamos la transacción con el baseline sintético del cliente.


### Recordemos lo construido hasta ahora (00 → 04)

In [ ]:
import os  # para leer variables de entorno (Colab Secrets ya copiados acá)
import copy  # para copiar documentos sin compartir referencias (ver InMemoryCollection)
import uuid  # para generar ids únicos de documentos insertados
from dataclasses import dataclass, field  # dataclass: clases con campos, sin escribir __init__ a mano
from typing import Any, Optional  # Optional[str] = "puede ser str o None"


@dataclass(frozen=True)  # frozen=True: una vez creado, Settings no se puede modificar
class Settings:
    """Configuración central del workshop, cargada desde variables de entorno o Colab Secrets."""
    mongodb_uri: Optional[str] = None  # None si no configuraste Atlas: usamos memoria local
    database_name: str = "fraud_mitigation_agent_workshop"  # nombre de la base en Atlas
    source_tag: str = "fraud_mitigation_agent_colab_workshop"  # para poder borrar solo lo nuestro
    embedding_dimensions: int = 8  # tamaño de los vectores que genera deterministic_embedding

    @classmethod
    def from_env(cls, **overrides):
        # Lee cada valor de una variable de entorno, con un default si no existe.
        values = {
            "mongodb_uri": os.getenv("MONGODB_URI"),
            "database_name": os.getenv("FRAUD_MITIGATION_AGENT_DATABASE", "fraud_mitigation_agent_workshop"),
            "source_tag": os.getenv("FRAUD_MITIGATION_AGENT_SOURCE_TAG", "fraud_mitigation_agent_colab_workshop"),
            "embedding_dimensions": int(os.getenv("FRAUD_MITIGATION_AGENT_EMBEDDING_DIMENSIONS", "8")),
        }
        values.update(overrides)  # permite pisar cualquier valor a mano, ej. Settings.from_env(database_name="test")
        return cls(**values)  # construye el Settings con esos valores


# --- Base de datos en memoria: imita la interfaz de pymongo (find_one, find,
# replace_one, delete_many, insert_one) para que el workshop corra sin Atlas. ---

def _matches(doc, query):
    # Recorre cada condición del query y las compara contra el documento.
    for key, expected in query.items():
        actual = doc.get(key)
        if isinstance(expected, dict):
            # Soporta los dos operadores de Mongo que este workshop necesita:
            # $exists (¿tiene o no tiene la clave?) y $in (¿está en esta lista?).
            if "$exists" in expected and (key in doc) != bool(expected["$exists"]):
                return False
            if "$in" in expected and actual not in expected["$in"]:
                return False
        elif actual != expected:
            # Caso simple: {"campo": valor} exige igualdad exacta.
            return False
    return True  # ninguna condición falló: el documento matchea


def _project(row, projection):
    item = copy.deepcopy(row)  # copia defensiva: nunca devolvemos el documento original
    if not projection:
        return item  # sin projection, se devuelve el documento completo
    excludes = [key for key, value in projection.items() if value == 0]  # claves a quitar
    includes = [key for key, value in projection.items() if value == 1]  # claves a conservar
    if includes:
        return {key: item[key] for key in includes if key in item}  # solo las incluidas
    for key in excludes:
        item.pop(key, None)  # saca las excluidas (típicamente {"_id": 0})
    return item


class _InsertResult:
    def __init__(self, inserted_id):
        self.inserted_id = inserted_id  # imita el objeto que devuelve pymongo.insert_one


class InMemoryCollection:
    def __init__(self):
        self.rows = []  # todos los documentos de esta "colección" viven en esta lista

    def find_one(self, query, projection=None):
        for row in self.rows:
            if _matches(row, query):
                return _project(row, projection)  # devuelve el primero que matchea
        return None  # ninguno matcheó

    def find(self, query=None, projection=None):
        query = query or {}  # sin query, devuelve todos los documentos
        return [_project(row, projection) for row in self.rows if _matches(row, query)]

    def replace_one(self, query, replacement, upsert=False):
        for i, row in enumerate(self.rows):
            if _matches(row, query):
                self.rows[i] = copy.deepcopy(replacement)  # ya existía: lo reemplaza
                return
        if upsert:
            self.rows.append(copy.deepcopy(replacement))  # no existía: lo agrega (upsert)

    def delete_many(self, query):
        self.rows = [row for row in self.rows if not _matches(row, query)]  # se queda con lo que NO matchea

    def insert_one(self, document):
        item = copy.deepcopy(document)  # copia defensiva del documento a insertar
        item.setdefault("_id", uuid.uuid4().hex)  # le pone un _id si no traía uno
        self.rows.append(item)
        return _InsertResult(item["_id"])

    def aggregate(self, pipeline):
        # $vectorSearch no está disponible en memoria local a propósito:
        # en el notebook 06 vamos a manejar esto con un fallback de similitud
        # coseno calculado en Python.
        raise RuntimeError("Atlas aggregation unavailable in local memory mode")


class InMemoryDB:
    """Imita `client[database_name]` / `db.coleccion` de pymongo, creando
    colecciones sobre la marcha la primera vez que se acceden."""

    def __init__(self):
        self._collections = {}  # diccionario nombre -> InMemoryCollection

    def __getitem__(self, name):
        return self.__getattr__(name)  # db["transactions"] hace lo mismo que db.transactions

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)  # evita interceptar atributos internos como _collections
        self._collections.setdefault(name, InMemoryCollection())  # la crea si es la primera vez
        return self._collections[name]


def get_client(uri, timeout_ms=10000):
    """Conecta a MongoDB Atlas real. Solo se usa si defines MONGODB_URI."""
    from pymongo import MongoClient  # import perezoso: solo hace falta si de verdad usás Atlas
    if not uri:
        raise ValueError("MONGODB_URI is required")
    client = MongoClient(uri, serverSelectionTimeoutMS=timeout_ms)
    client.admin.command("ping")  # falla rápido acá si la conexión no funciona
    return client


def get_database(client, database_name):
    return client[database_name]  # selecciona (o crea) la base dentro del cluster


import hashlib  # para generar un hash reproducible de cada palabra
import re  # para separar el texto en palabras (tokens)
import numpy as np  # para operar con vectores (sumas, norma)


def _token_value(token, dimensions):
    digest = hashlib.sha256(token.encode("utf-8")).digest()  # 32 bytes, siempre iguales para el mismo token
    values = np.frombuffer(digest, dtype=np.uint8)[:dimensions].astype(float)  # toma los primeros `dimensions` bytes
    return (values / 127.5) - 1.0  # reescala de [0, 255] a, aproximadamente, [-1, 1]


def deterministic_embedding(text, dimensions=8):
    """Embedding determinístico para el workshop (hashing, sin modelo ni red).
    Lo usamos desde ya para poder sembrar los datos de ejemplo; en el
    notebook 06 vamos a entender cómo funciona y a construir búsqueda por
    similitud vectorial sobre él."""
    tokens = re.findall(r"[a-zA-Z0-9_áéíóúñ-]+", (text or "").lower())  # separa el texto en palabras, en minúscula
    if not tokens:
        return [0.0] * dimensions  # texto vacío -> vector de ceros
    vector = np.zeros(dimensions, dtype=float)  # arranca en cero
    for token in tokens:
        vector += _token_value(token, dimensions)  # suma el vector de cada palabra
    norm = np.linalg.norm(vector)  # longitud del vector resultante
    if norm == 0:
        return [0.0] * dimensions  # evita dividir por cero
    return (vector / norm).round(6).tolist()  # normaliza (longitud 1) y lo convierte a lista de Python


def _pattern(tx_id, fraud_type, text):
    return {
        "tx_id": tx_id,
        "fraud_confirmed": True,
        "fraud_type": fraud_type,
        "fraud_signature_text": text,
        "embedding": deterministic_embedding(text),
        "source_tag": "fraud_mitigation_agent_synthetic",
    }


def demo_documents():
    # Cuatro "firmas" de fraude conocidas: la colección fraud_patterns que la
    # búsqueda por similitud vectorial va a comparar contra cada transacción nueva.
    patterns = [
        _pattern("pattern-001", "account_takeover", "new device new ip impossible travel odd hour credential reset high amount"),
        _pattern("pattern-002", "card_testing", "many small attempts new ip repeated velocity web checkout"),
        _pattern("pattern-003", "synthetic_identity", "new customer device mismatch unusual geo high amount mobile"),
        _pattern("pattern-004", "money_mule", "rapid transfer beneficiary new device distant geo unusual hour"),
    ]
    # Tres arquetipos de cliente, cada uno emparejado con una transacción que
    # se espera que caiga en una banda de decisión distinta (APPROVE/STEP-UP/DENY).
    customers = [
        {
            "customer_id": "customer-normal",
            "usual_ips": ["198.51.100.10"],
            "usual_devices": ["device-normal-001"],
            "usual_countries": ["MX"],
            "avg_amount": 1200,
            "p95_amount": 4500,
            "transactions_24h": 3,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "customer_id": "customer-stepup",
            "usual_ips": ["198.51.100.20"],
            "usual_devices": ["device-step-001"],
            "usual_countries": ["MX"],
            "avg_amount": 1800,
            "p95_amount": 9000,
            "transactions_24h": 4,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "customer_id": "customer-risky",
            "usual_ips": ["198.51.100.30"],
            "usual_devices": ["device-risky-001"],
            "usual_countries": ["MX"],
            "avg_amount": 2500,
            "p95_amount": 12000,
            "transactions_24h": 2,
            "transactions_10m": 1,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
    ]
    # tx-normal-001 -> se espera APPROVE, tx-stepup-001 -> se espera STEP-UP,
    # tx-risky-001 -> se espera DENY. Los usamos en todos los notebooks.
    transactions = [
        {
            "tx_id": "tx-normal-001", "customer_id": "customer-normal", "amount": 850,
            "currency": "MXN", "timestamp": "2025-01-15T16:20:00Z", "channel": "web",
            "ip": "198.51.100.10", "device_id": "device-normal-001", "geo_km_from_usual": 2,
            "ground_truth_fraud": False, "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "tx_id": "tx-stepup-001", "customer_id": "customer-stepup", "amount": 12000,
            "currency": "MXN", "timestamp": "2025-01-15T22:40:00Z", "channel": "mobile",
            "ip": "198.51.100.20", "device_id": "device-step-001", "geo_km_from_usual": 120,
            "ground_truth_fraud": False, "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "tx_id": "tx-risky-001", "customer_id": "customer-risky", "amount": 5000000,
            "currency": "COP", "timestamp": "2025-01-15T03:00:00Z", "channel": "mobile",
            "ip": "203.0.113.30", "device_id": "device-new-003", "geo_km_from_usual": 9000,
            "ground_truth_fraud": True, "source_tag": "fraud_mitigation_agent_synthetic",
        },
    ]
    rules = {
        "config_id": "risk_rules_config",
        "version": "demo-v1",
        "enabled": True,
        "thresholds": {
            "high_amount": 1000000,
            "amount_multiplier": 5,
            "impossible_travel_km": 500,
            "velocity_10m": 5,
        },
        "weights": {"vector": 0.40, "signals": 0.35, "rules": 0.25},
        "decision_thresholds": {"approve_max": 39, "step_up_max": 69},
        "source_tag": "fraud_mitigation_agent_synthetic",
    }
    return {"patterns": patterns, "customers": customers, "transactions": transactions, "rules": rules}


def seed_demo_data(db, reset=False):
    """Carga los datos sintéticos en la base (Atlas o InMemoryDB). Es idempotente:
    replace_one(upsert=True) evita duplicados si vuelves a correr esta celda."""
    docs = demo_documents()  # arma los 4 patrones, 3 clientes, 3 transacciones y las reglas
    collections = {  # atajos a cada colección de destino
        "patterns": db["fraud_patterns"],
        "customers": db["customer_state"],
        "transactions": db["transactions"],
        "rules": db["risk_rules_config"],
    }
    if reset:
        for collection in collections.values():
            # Borra solo lo que sembramos nosotros (por source_tag), nunca datos ajenos.
            collection.delete_many({"source_tag": {"$in": ["fraud_mitigation_agent_synthetic", "fraud_mitigation_agent_colab_workshop"]}})
    for document in docs["patterns"]:
        collections["patterns"].replace_one({"tx_id": document["tx_id"]}, document, upsert=True)  # inserta o actualiza
    for document in docs["customers"]:
        collections["customers"].replace_one({"customer_id": document["customer_id"]}, document, upsert=True)
    for document in docs["transactions"]:
        document = dict(document)  # copia: no modificamos el diccionario original de demo_documents()
        document["fraud_signature_text"] = (
            "new device new ip impossible travel odd hour high amount"
            if document["ground_truth_fraud"] else  # las transacciones fraudulentas comparten esta descripción...
            "familiar device familiar ip normal amount"  # ...y las normales, esta otra
        )
        document["embedding"] = deterministic_embedding(document["fraud_signature_text"])  # vector para vector search
        collections["transactions"].replace_one({"tx_id": document["tx_id"]}, document, upsert=True)
    collections["rules"].replace_one({"config_id": docs["rules"]["config_id"]}, docs["rules"], upsert=True)
    # Resumen de cuántos documentos se sembraron en cada colección (listas) o 1 (las reglas, un solo documento).
    return {key: len(value) if isinstance(value, list) else 1 for key, value in docs.items()}


# Los Secrets de Colab no se inyectan solos como variables de entorno: hay
# que leerlos explícitamente con userdata.get(...) y copiarlos a os.environ.
# Cada clave se intenta por separado para que una que no exista (userdata.get
# lanza una excepción, no devuelve None) no tumbe la lectura de las demás.
try:
    from google.colab import userdata
except Exception:
    userdata = None

if userdata is not None:
    for _secret_name in ("MONGODB_URI", "LLM_API_KEY", "LLM_MODEL", "LLM_BASE_URL"):
        try:
            _secret_value = userdata.get(_secret_name)
        except Exception:
            _secret_value = None
        if _secret_value:
            os.environ[_secret_name] = _secret_value

settings = Settings.from_env()
if settings.mongodb_uri:
    # pymongo no viene preinstalado en este notebook (a diferencia de 00, que
    # sí lo instala siempre): solo lo instalamos aquí, justo a tiempo, si de
    # verdad vas a usar Atlas real. El camino en memoria no lo necesita.
    try:
        import pymongo  # noqa: F401
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo[srv]"], check=True)
    client = get_client(settings.mongodb_uri)
    db = get_database(client, settings.database_name)
else:
    db = InMemoryDB()
seed_demo_data(db, reset=False)
print("Runtime listo:", type(db).__name__)


class MockLLMProvider:
    """Proveedor offline para Colab: determinístico, sin red y sin API key."""

    def complete(self, prompt, system=None):
        # Coincidencia de palabras clave en vez de un modelo real: alcanza
        # para demostrar la mecánica del agente sin necesitar ninguna API key.
        text = (prompt or "").lower()  # normaliza el prompt para comparar en minúscula
        if "fraud mitigation agent" in text or "workshop" in text:
            return "El Fraud Mitigation Agent separa herramientas de contexto, scoring determinístico y decisión auditable."
        if "fraude" in text or "fraud" in text:
            return "El flujo combina reglas, señales de comportamiento y similitud vectorial; el resultado final no depende de una respuesta libre del LLM."
        return "MockLLMProvider: respuesta local reproducible para el workshop."  # respuesta genérica de reserva


class OpenAICompatibleProvider:
    """Adaptador opcional: nunca lo exige el camino core. Funciona con la API
    real de OpenAI y con cualquier otro servicio que exponga un endpoint
    compatible con /chat/completions (Groq, NVIDIA NIM, Google AI Studio, un
    servidor local, etc.) — cambiar de proveedor es solo otro
    base_url/api_key/model, sin tocar código. Ver README.md para opciones
    gratuitas sin tarjeta de crédito."""

    def __init__(self, api_key, model, base_url=None):
        if not api_key:
            raise ValueError("LLM_API_KEY is required for the optional provider")
        if not model:
            raise ValueError("LLM_MODEL is required for the optional provider")
        from openai import OpenAI  # import perezoso: solo hace falta si de verdad usás este proveedor
        kwargs = {"api_key": api_key}
        if base_url:
            kwargs["base_url"] = base_url  # sin esto, apunta por defecto a la API de OpenAI
        self.client = OpenAI(**kwargs)
        self.model = model

    def complete(self, prompt, system=None):
        messages = []  # el chat se arma como una lista de mensajes con rol
        if system:
            messages.append({"role": "system", "content": system})  # instrucciones para el modelo
        messages.append({"role": "user", "content": prompt})  # la pregunta o pedido en sí
        # temperature=0: respuestas lo más reproducibles posible (nunca 100%, pero se acerca).
        response = self.client.chat.completions.create(model=self.model, messages=messages, temperature=0)
        return response.choices[0].message.content or ""  # texto de la primera respuesta


# `pip install` de las dos librerías nuevas de esta etapa. langchain-core trae
# el decorador @tool (para describir una función como una "herramienta" que
# un framework de agentes puede entender); langgraph trae StateGraph (el
# motor que ejecuta los nodos en el orden que nosotros definamos).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langchain-core", "langgraph"], check=True)

# time: lo usamos para medir cuánto tarda cada Tool (igual que antes).
import time
# TypedDict describe la FORMA de un diccionario (qué claves tiene y de qué
# tipo es cada una) sin crear una clase nueva de verdad — es solo para que
# el editor y LangGraph sepan qué esperar en el "estado" del grafo.
# Annotated nos deja agregarle una instrucción extra a un tipo (ver más abajo).
from typing import TypedDict, Annotated
# operator.add es la función que suma (o concatena) dos valores con "+".
# La vamos a usar para decirle a LangGraph "concatená las listas, no las
# reemplaces" (ver el campo `trace` de AgentState).
import operator
# @tool: el decorador de LangChain que convierte una función Python normal
# en una Tool con nombre, descripción y "args schema" (qué argumentos recibe
# y de qué tipo), inferidos automáticamente del docstring y los type hints.
from langchain_core.tools import tool
# StateGraph: el grafo de LangGraph. START y END son los marcadores
# especiales de "por acá entra" y "por acá termina" el grafo.
from langgraph.graph import StateGraph, START, END


class ToolResult:
    """Envoltorio uniforme para el resultado de cada Tool, igual que antes de
    usar LangChain: nombre, estado, datos, latencia y error. Lo seguimos
    usando nosotros (no es parte de LangChain) porque nos conviene tener
    siempre la misma forma en la traza, sin importar qué Tool se llamó."""

    def __init__(self, tool_name, status="success", data=None, latency_ms=0.0, error=None):
        self.tool_name = tool_name  # cuál Tool generó este resultado
        self.status = status  # "success" o "error"
        self.data = data  # lo que devolvió la Tool, si no falló
        self.latency_ms = latency_ms  # cuánto tardó la llamada
        self.error = error  # mensaje de la excepción, si falló

    def as_dict(self):
        return {  # forma que va a tener cada entrada de la traza
            "tool_name": self.tool_name,
            "status": self.status,
            "data": self.data,
            "latency_ms": round(self.latency_ms, 2),
            "error": self.error,
        }


def run_tool(name, langchain_tool, tool_input):
    """Invoca una Tool de LangChain y envuelve el resultado en un ToolResult.

    `langchain_tool` es un objeto creado con @tool (no una función normal):
    por eso no se llama como `langchain_tool(...)`, sino con
    `.invoke(tool_input)` — así es como LangChain espera que se llame
    cualquier Tool, sea quien sea el que decide invocarla (un LLM, o
    nosotros mismos, como en este workshop).

    `tool_input` siempre es un diccionario, por ejemplo
    {"transaction_id": "tx-1"}: las claves deben coincidir con los nombres
    de los parámetros de la función original.

    Igual que antes: medimos el tiempo y atrapamos cualquier excepción, para
    que una Tool que falla nunca tumbe el resto del agente — la falla queda
    registrada como status="error" en la traza en vez de romper el programa."""
    started = time.perf_counter()  # marca de tiempo antes de llamar la Tool
    try:
        # .invoke() ejecuta la función real "de adentro" de la Tool y
        # devuelve lo que esa función haya hecho return.
        result = langchain_tool.invoke(tool_input)
        elapsed_ms = (time.perf_counter() - started) * 1000  # cuánto tardó, en milisegundos
        return ToolResult(name, data=result, latency_ms=elapsed_ms)  # éxito
    except Exception as exc:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return ToolResult(name, status="error", error=str(exc), latency_ms=elapsed_ms)  # falla atrapada


def make_get_transaction_tool(db):
    """"Fábrica" de la Tool: recibe la base de datos UNA vez y devuelve una
    función ya decorada con @tool que la "recuerda" (esto se llama closure).
    Así, el argumento de la Tool es solo lo que hace falta para llamarla
    (transaction_id) — `db` queda escondida adentro, no forma parte de lo
    que un LLM (o nuestro propio código) tendría que especificar."""

    @tool
    def get_transaction(transaction_id: str) -> dict:
        """Obtiene la transacción a evaluar por su tx_id."""
        # Este docstring no es solo un comentario: LangChain lo usa como la
        # "description" de la Tool. Y el type hint `transaction_id: str` es
        # lo que arma su "args schema" (qué argumentos acepta y de qué
        # tipo). Por eso una Tool de LangChain es autodescriptiva.
        document = db.transactions.find_one({"tx_id": transaction_id}, {"_id": 0})  # {"_id": 0}: no lo necesitamos
        if not document:
            # No atrapamos esta excepción acá: run_tool() de la celda
            # anterior es quien la convierte en un ToolResult con error.
            raise LookupError(f"Transaction not found: {transaction_id}")
        return document

    return get_transaction


def make_get_customer_state_tool(db):
    """Misma idea que la Tool anterior: una fábrica que "recuerda" `db` y
    devuelve una Tool que solo necesita el customer_id como argumento."""

    @tool
    def get_customer_state(customer_id: str) -> dict:
        """Obtiene el estado base conocido del cliente (dispositivos, IPs,
        montos habituales). Convierte una transacción aislada en algo que se
        puede comparar contra "lo normal para este cliente"."""
        document = db.customer_state.find_one({"customer_id": customer_id}, {"_id": 0})  # {"_id": 0}: no lo necesitamos
        if not document:
            raise LookupError(f"Customer state not found: {customer_id}")  # atrapada por run_tool
        return document

    return get_customer_state


def make_evaluate_rules_tool(db):
    """Esta Tool sí necesita `db` (para leer los umbrales configurados), así
    que también usamos el patrón de fábrica: cerramos sobre `db` y devolvemos
    la Tool ya lista para invocar solo con transaction/customer_state."""

    @tool
    def evaluate_rules(transaction: dict, customer_state: dict) -> dict:
        """Evalúa reglas con umbrales configurables (guardados en
        risk_rules_config, no hard-codeados en el prompt ni en esta función)."""
        config = db.risk_rules_config.find_one({"config_id": "risk_rules_config"}, {"_id": 0}) or {}  # los umbrales sembrados en 00
        thresholds = config.get("thresholds", {})  # atajo a la sub-sección de umbrales
        state = customer_state or {}  # por si llega None
        triggered = []  # acá se van acumulando las reglas que disparan
        amount = float(transaction.get("amount", 0))
        if amount >= thresholds.get("high_amount", float("inf")):  # sin threshold configurado, nunca dispara
            triggered.append({"code": "high_amount", "severity": "high", "detail": amount})
        # "Nuevo" significa "no está en el set conocido de este cliente": un
        # cliente sin historial (usual_ips/usual_devices vacíos) siempre
        # dispara estas dos reglas, por diseño.
        if transaction.get("ip") not in state.get("usual_ips", []):
            triggered.append({"code": "new_ip", "severity": "medium", "detail": transaction.get("ip")})
        if transaction.get("device_id") not in state.get("usual_devices", []):
            triggered.append({"code": "new_device", "severity": "high", "detail": transaction.get("device_id")})
        if float(transaction.get("geo_km_from_usual", 0)) >= thresholds.get("impossible_travel_km", float("inf")):
            triggered.append({"code": "impossible_travel", "severity": "critical", "detail": transaction.get("geo_km_from_usual")})
        # Extrae la hora directo del timestamp ISO (ej. "...T03:00:00Z" -> 3);
        # a propósito no maneja zonas horarias, para mantenerlo simple.
        hour = int(transaction.get("timestamp", "T12:").split("T")[-1][:2] or 12)
        if hour < 6 or hour >= 23:
            triggered.append({"code": "odd_hour", "severity": "medium", "detail": hour})
        # Devuelve también config_version y weights: score_and_decide los va
        # a necesitar más adelante, así no hace falta volver a leer la config.
        return {"triggered_rules": triggered, "config_version": config.get("version", "unknown"), "weights": config.get("weights", {})}

    return evaluate_rules


### Nueva pieza: análisis de comportamiento

`analyze_behavior` es distinto de las reglas: en vez de umbrales fijos para todos, mide desviaciones relativas al historial propio del cliente (por ejemplo, "5 veces el promedio de este cliente"). A diferencia de las Tools anteriores, esta no necesita la base de datos, así que su fábrica no recibe `db`.

In [ ]:
def make_analyze_behavior_tool():
    """Esta Tool NO toca la base de datos (solo compara números que ya
    tenemos en memoria), así que no necesita cerrar sobre `db` — por eso su
    fábrica no recibe ningún argumento. No todas las Tools son iguales: cada
    una usa exactamente lo que necesita y nada más."""

    @tool
    def analyze_behavior(transaction: dict, customer_state: dict) -> dict:
        """Detecta desviaciones estadísticas respecto al historial propio del
        cliente (a diferencia de evaluate_rules, que usa umbrales fijos
        iguales para todos los clientes)."""
        signals = []  # acá se van acumulando las señales de comportamiento detectadas
        amount = float(transaction.get("amount", 0))
        average = float(customer_state.get("avg_amount", 0))
        # Guard `average`: un cliente nuevo con avg_amount=0 no tiene
        # baseline del cual desviarse, así que se omite en vez de marcar todo.
        if average and amount > average * 5:
            signals.append({"code": "amount_deviation", "score": 35, "detail": f"{amount} > 5x average {average}"})
        if transaction.get("ip") not in customer_state.get("usual_ips", []):
            signals.append({"code": "ip_deviation", "score": 20, "detail": "IP outside usual set"})
        if transaction.get("device_id") not in customer_state.get("usual_devices", []):
            signals.append({"code": "device_deviation", "score": 25, "detail": "device outside usual set"})
        geo = float(transaction.get("geo_km_from_usual", 0))
        if geo > 500:  # más de 500 km del lugar habitual
            signals.append({"code": "geo_deviation", "score": 25, "detail": f"{geo} km"})
        # baseline: se incluye para que quede registrado contra qué se comparó.
        return {"signals": signals, "baseline": {"avg_amount": average, "p95_amount": customer_state.get("p95_amount")}}

    return analyze_behavior


### El grafo y el agente (etapa 5 de 8): suma análisis de comportamiento

In [ ]:
class AgentState(TypedDict, total=False):
    """Suma el campo `behavior_result`."""
    transaction_id: str  # lo pasamos nosotros al invocar el grafo (.invoke)
    transaction: dict  # lo llena el nodo get_transaction
    customer: dict  # lo llena el nodo get_customer_state
    rules_result: dict  # lo llena el nodo evaluate_rules
    behavior_result: dict  # nuevo: lo llena el nodo analyze_behavior
    trace: Annotated[list, operator.add]  # se concatena, nunca se pisa
    error: str  # motivo de la primera falla, si la hay


def build_graph(db):
    """Cuarto nodo: analyze_behavior. Nota que su Tool no recibe `db`
    (make_analyze_behavior_tool() se llama sin argumentos)."""
    get_transaction_tool = make_get_transaction_tool(db)  # Tool del nodo 1
    get_customer_state_tool = make_get_customer_state_tool(db)  # Tool del nodo 2
    evaluate_rules_tool = make_evaluate_rules_tool(db)  # Tool del nodo 3
    analyze_behavior_tool = make_analyze_behavior_tool()  # Tool del nodo 4 (nueva, sin db)

    def node_get_transaction(state: AgentState) -> dict:
        # Armamos el diccionario de entrada (así se invoca una @tool) y
        # llamamos la Tool a través de run_tool, que mide el tiempo y
        # atrapa cualquier excepción por nosotros.
        result = run_tool("get_transaction", get_transaction_tool, {"transaction_id": state["transaction_id"]})
        # Si la Tool falló (por ejemplo, tx_id no existe), guardamos el
        # motivo en `error`; los nodos siguientes lo van a detectar.
        if result.status != "success":
            return {"trace": [result.as_dict()], "error": result.error}
        # Éxito: agregamos la traza de este paso y la transacción obtenida
        # al estado, para que los próximos nodos puedan usarla.
        return {"trace": [result.as_dict()], "transaction": result.data}

    def node_get_customer_state(state: AgentState) -> dict:
        # Guard: si get_transaction (el paso anterior) ya falló, no seguimos
        # llamando Tools — devolvemos un diccionario vacío (sin cambios) y
        # dejamos que el grafo llegue a END con el error ya guardado.
        if state.get("error"):
            return {}
        # Sacamos el customer_id de la transacción que trajo el nodo anterior.
        customer_id = state["transaction"]["customer_id"]
        result = run_tool("get_customer_state", get_customer_state_tool, {"customer_id": customer_id})
        if result.status != "success":
            # Un cliente sin historial (o no encontrado) no frena el
            # análisis: seguimos con un baseline vacío, no con un error.
            return {"trace": [result.as_dict()], "customer": {}}
        return {"trace": [result.as_dict()], "customer": result.data}

    def node_evaluate_rules(state: AgentState) -> dict:
        if state.get("error"):
            return {}
        # Esta Tool recibe DOS argumentos (transaction y customer_state): el
        # diccionario de tool_input simplemente lleva las dos claves.
        tool_input = {"transaction": state["transaction"], "customer_state": state.get("customer", {})}
        # evaluate_rules_tool ya "recuerda" `db` (se armó con make_evaluate_rules_tool(db)).
        result = run_tool("evaluate_rules", evaluate_rules_tool, tool_input)
        if result.status != "success":
            return {"trace": [result.as_dict()], "error": result.error}
        # Guardamos el resultado en `rules_result`, la clave que van a leer
        # los nodos de scoring más adelante en el grafo.
        return {"trace": [result.as_dict()], "rules_result": result.data}

    def node_analyze_behavior(state: AgentState) -> dict:
        if state.get("error"):
            return {}
        # Mismos argumentos que evaluate_rules: transacción + contexto del cliente.
        tool_input = {"transaction": state["transaction"], "customer_state": state.get("customer", {})}
        # analyze_behavior_tool no necesita `db` (se construyó sin argumentos
        # con make_analyze_behavior_tool()), pero se invoca igual que las demás.
        result = run_tool("analyze_behavior", analyze_behavior_tool, tool_input)
        if result.status != "success":
            return {"trace": [result.as_dict()], "error": result.error}
        # Guardamos las señales de comportamiento en `behavior_result`.
        return {"trace": [result.as_dict()], "behavior_result": result.data}

    graph = StateGraph(AgentState)  # crea el grafo con el esquema AgentState
    graph.add_node("get_transaction", node_get_transaction)  # nodo 1
    graph.add_node("get_customer_state", node_get_customer_state)  # nodo 2
    graph.add_node("evaluate_rules", node_evaluate_rules)  # nodo 3
    graph.add_node("analyze_behavior", node_analyze_behavior)  # nodo 4 (nuevo)
    graph.add_edge(START, "get_transaction")  # por dónde arranca el grafo
    graph.add_edge("get_transaction", "get_customer_state")  # 1 -> 2
    graph.add_edge("get_customer_state", "evaluate_rules")  # 2 -> 3
    graph.add_edge("evaluate_rules", "analyze_behavior")  # 3 -> 4
    graph.add_edge("analyze_behavior", END)  # 4 -> fin
    return graph.compile()  # arma el grafo ejecutable (con .invoke)


class FraudAgent:
    """Etapa 5 de 8: se suma el análisis de comportamiento."""

    def __init__(self, db, provider=None):
        self.db = db  # guardamos la base para poder usarla más adelante si hace falta
        self.provider = provider or MockLLMProvider()  # MockLLMProvider si no nos pasan uno
        self.graph = build_graph(db)  # arma el grafo UNA vez, no en cada análisis

    def answer(self, prompt):
        # Ruta libre (sin grafo): solo le pasamos el prompt al proveedor de LLM.
        return self.provider.complete(prompt, system="You are the Fraud Mitigation Agent, a concise workshop assistant.")

    def analyze(self, transaction_id):
        # .invoke(...) corre el grafo de punta a punta con este estado inicial.
        state = self.graph.invoke({"transaction_id": transaction_id, "trace": []})
        return {
            "trace": state["trace"],  # traza acumulada de los 4 nodos
            "transaction": state.get("transaction"),  # lo puso get_transaction
            "customer": state.get("customer"),  # lo puso get_customer_state
            "rules": state.get("rules_result"),  # lo puso evaluate_rules
            "behavior": state.get("behavior_result"),  # lo puso analyze_behavior (nuevo)
            "error": state.get("error"),  # si algún nodo falló
        }


### Probemos

In [ ]:
# Igual que en el notebook 01: si configuraste LLM_API_KEY y LLM_MODEL (Colab
# Secret o variable de entorno) con alguno de los proveedores gratuitos del
# README, usamos un LLM real para la explicación final en vez de
# MockLLMProvider. El workshop nunca lo exige, así que si no están
# configuradas seguimos con el mock sin romper nada.
if os.getenv("LLM_API_KEY") and os.getenv("LLM_MODEL"):
    agent = FraudAgent(db, provider=OpenAICompatibleProvider(os.getenv("LLM_API_KEY"), os.getenv("LLM_MODEL"), os.getenv("LLM_BASE_URL")))
else:
    agent = FraudAgent(db)
import json
result = agent.analyze("tx-risky-001")
print(json.dumps(result["behavior"], indent=2, default=str))
